# LRASPP-MobileNetV3 fine-tuning
        **vision › segmentation** · notebook `lraspp_finetune`

        **Kapan dipakai:** Segmentasi ringan dan cepat dengan transfer learning; titik awal segmentasi di laptop.

        **Jebakan umum:** Kepala diganti 2 kelas; resolusi 256px menghilangkan detail halus.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** Penn-Fudan (170 gambar pejalan kaki, kotak + mask); 120 latih / 50 uji tetap. Tugas: segmentasi biner (orang vs latar). Metrik: **IoU** dan **Dice**.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
MODEL_KEY = "lraspp_finetune"
OUT = out_dir("vision", "segmentation", MODEL_KEY)
SEED = 42; set_seed(SEED)
SIZE, EPOCHS, LR = 256, 12, 0.001

## 2. Data: gambar dan mask orang, diubah ke SIZE×SIZE

In [ ]:
from pathlib import Path
import torch
from superdb.vision_det import load_pennfudan, split_idx, seg_metrics, seg_model, to_binary_head, train_segmenter, predict_masks, unet
from superdb.vision import device
D = load_pennfudan(size=SIZE); itr, ite = split_idx(len(D), 50)
X = np.stack([d["img"] for d in D]); M = np.stack([d["mask"] for d in D])
print(X.shape, M.shape, f"| porsi piksel orang: {M.mean():.1%} | device: {device()}")
fig, ax = plt.subplots(1, 4, figsize=(10, 2.8))
for k in range(2): ax[2 * k].imshow(X[k]); ax[2 * k + 1].imshow(M[k], cmap="gray"); ax[2 * k].axis("off"); ax[2 * k + 1].axis("off")
plt.tight_layout(); plt.show()

## 3. Model

In [ ]:
model = to_binary_head(seg_model("lraspp", pretrained=True), "lraspp"); imagenet_norm = True      # kepala diganti 2 kelas
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} juta parameter")

## 4. Latih (IoU validation per epoch)

In [ ]:
hist = train_segmenter(model, X[itr], M[itr], X[ite], M[ite], epochs=EPOCHS, lr=LR, batch=8, size=SIZE, seed=SEED, imagenet_norm=imagenet_norm)
h = pd.DataFrame(hist); display(h.round(4)); h.val_IoU.plot(marker="o", title="IoU validation per epoch"); plt.tight_layout(); plt.savefig(OUT / "curve.png", dpi=130); plt.show()
pred = predict_masks(model, X[ite], SIZE, imagenet_norm=imagenet_norm); metrics = seg_metrics(pred, M[ite]); display(show_table(metrics))

## 5. Lihat hasil (gambar | mask asli | prediksi)

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(7, 7))
for r, k in enumerate((0, 1, 2)):
    ax[r, 0].imshow(X[ite[k]]); ax[r, 1].imshow(M[ite[k]], cmap="gray"); ax[r, 2].imshow(pred[k], cmap="gray")
    [a.axis("off") for a in ax[r]]
ax[0, 0].set_title("gambar"); ax[0, 1].set_title("mask asli"); ax[0, 2].set_title("prediksi")
plt.tight_layout(); plt.savefig(OUT / "segmentation.png", dpi=130); plt.show()
save_json({"metrics": metrics, "history": hist}, OUT / "metrics.json")

In [ ]:
torch.save(model.state_dict(), OUT / "model.pt"); sd = torch.load(OUT / "model.pt", map_location="cpu"); model.load_state_dict(sd)
agree = float((predict_masks(model, X[ite[:4]], SIZE, imagenet_norm=imagenet_norm) == pred[:4]).mean())
assert agree > 0.99, f"model muat-ulang berbeda (kesepakatan piksel {agree:.4f})"; print(f"OK. model tersimpan; kesepakatan piksel setelah muat ulang = {agree:.4f}")

## Langkah berikutnya
- Naikkan `SIZE` (384/512) untuk batas objek lebih halus; tambahkan augmentasi (flip sudah ada).
- Loss lain untuk objek kecil/kelas tak seimbang: Dice/Focal.
- Multi-kelas: ganti 2 dengan jumlah kelas dan mask berindeks kelas.